# Sinusoidal Positional Encoding from Scratch

This OOP notebook loads the previously trained English and Bangla embedding matrices and tokenized data from Google Drive, creates sine/cosine positional encodings for any embedding dimension, adds them to token vectors, and saves the results to Drive.

## Imports

Imports configuration, JSON, mathematics, PyTorch, and data-loading utilities.

In [1]:
import json
import math
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

## Mount Google Drive

Connects Colab to the Drive folders created by the tokenization and embedding notebooks.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Configuration
128D embeddings, sequence length, batch size, and output precision.

In [3]:
@dataclass
class PositionalEncodingConfig:
    drive_root: str = "/content/drive/MyDrive/Transformer"
    tokenized_folder: str = "tokenized_data"
    trained_embedding_folder: str = "trained_embeddings"
    output_folder: str = "position_encoded_embeddings"

    # Must match the dimension trained in the previous notebook.
    embedding_dimension: int = 128
    max_sequence_length: int = 32
    batch_size: int = 128
    padding_id: int = 0
    scale_token_embeddings: bool = True
    save_as_float16: bool = True


config = PositionalEncodingConfig()

if config.embedding_dimension not in {64, 128}:
    raise ValueError("embedding_dimension must be 64 or 128.")
if config.max_sequence_length <= 0:
    raise ValueError("max_sequence_length must be positive.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
drive_root = Path(config.drive_root)
tokenized_dir = drive_root / config.tokenized_folder
embedding_dir = (
    drive_root
    / config.trained_embedding_folder
    / f"{config.embedding_dimension}d"
)
output_dir = (
    drive_root
    / config.output_folder
    / f"{config.embedding_dimension}d"
)
output_dir.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Embedding input:", embedding_dir)
print("Tokenized input:", tokenized_dir)
print("Output:", output_dir)

Device: cuda
Embedding input: /content/drive/MyDrive/Transformer/trained_embeddings/128d
Tokenized input: /content/drive/MyDrive/Transformer/tokenized_data
Output: /content/drive/MyDrive/Transformer/position_encoded_embeddings/128d


## Validate Drive files

Checks that both saved embedding matrices and all three tokenized splits exist.

In [4]:
english_embedding_path = embedding_dir / "english_embedding_matrix.pt"
bangla_embedding_path = embedding_dir / "bangla_embedding_matrix.pt"

tokenized_paths = {
    split_name: tokenized_dir / f"{split_name}_tokenized.jsonl"
    for split_name in ("train", "validation", "test")
}

required_paths = {
    "English embedding matrix": english_embedding_path,
    "Bangla embedding matrix": bangla_embedding_path,
    **{
        f"{split_name} tokenization": path
        for split_name, path in tokenized_paths.items()
    },
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Drive files: " + ", ".join(missing)
    )

print("All required Drive files were found.")

All required Drive files were found.


## Drive embedding store

Loads the English and Bangla matrices from Drive and verifies their dimensions.

In [6]:
class DriveEmbeddingStore:
    def __init__(self, english_path, bangla_path, expected_dimension):
        self.english_path = Path(english_path)
        self.bangla_path = Path(bangla_path)
        self.expected_dimension = expected_dimension
        self.english_matrix = None
        self.bangla_matrix = None

    @staticmethod
    def _load_tensor(path):
        try:
            tensor = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            tensor = torch.load(path, map_location="cpu")

        if not isinstance(tensor, torch.Tensor) or tensor.ndim != 2:
            raise TypeError(f"{path} must contain one 2D tensor.")
        return tensor.float()

    def load(self):
        self.english_matrix = self._load_tensor(self.english_path)
        self.bangla_matrix = self._load_tensor(self.bangla_path)

        for language, matrix in {
            "English": self.english_matrix,
            "Bangla": self.bangla_matrix,
        }.items():
            if matrix.shape[1] != self.expected_dimension:
                raise ValueError(
                    f"{language} matrix dimension is {matrix.shape[1]}, "
                    f"but config requests {self.expected_dimension}."
                )

        return self


embedding_store = DriveEmbeddingStore(
    english_embedding_path,
    bangla_embedding_path,
    config.embedding_dimension,
).load()

print("English matrix:", tuple(embedding_store.english_matrix.shape))
print("Bangla matrix:", tuple(embedding_store.bangla_matrix.shape))

English matrix: (449, 128)
Bangla matrix: (479, 128)


## Load tokenized JSONL data

Reads the saved token IDs without rebuilding or changing the vocabularies.

In [7]:
def load_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number} of {path}"
                ) from error
    return records


tokenized_records = {
    split_name: load_jsonl(path)
    for split_name, path in tokenized_paths.items()
}

for split_name, records in tokenized_records.items():
    print(split_name, len(records))

train 1800
validation 100
test 100


## Fixed-length token dataset

Truncates or pads source and target ID sequences and creates masks where `True` means padding.

In [8]:
class FixedLengthTokenDataset(Dataset):
    def __init__(self, records, max_length, padding_id=0):
        self.records = records
        self.max_length = max_length
        self.padding_id = padding_id

    def __len__(self):
        return len(self.records)

    def _pad_or_truncate(self, token_ids):
        token_ids = [int(token_id) for token_id in token_ids[: self.max_length]]
        padding_needed = self.max_length - len(token_ids)
        token_ids.extend([self.padding_id] * padding_needed)

        ids = torch.tensor(token_ids, dtype=torch.long)
        padding_mask = ids.eq(self.padding_id)
        return ids, padding_mask

    def __getitem__(self, index):
        record = self.records[index]
        source_ids, source_mask = self._pad_or_truncate(record["source_ids"])
        target_ids, target_mask = self._pad_or_truncate(record["target_ids"])

        return {
            "source_ids": source_ids,
            "target_ids": target_ids,
            "source_padding_mask": source_mask,
            "target_padding_mask": target_mask,
        }

## Token embedding lookup

Wraps each Drive-loaded matrix in an OOP PyTorch layer and optionally applies standard Transformer scaling.

In [9]:
class PretrainedTokenEmbedding(nn.Module):
    def __init__(self, embedding_matrix, trainable=False, scale=True):
        super().__init__()
        self.embedding_dimension = embedding_matrix.shape[1]
        self.scale = scale
        self.weight = nn.Parameter(
            embedding_matrix.clone(),
            requires_grad=trainable,
        )

    def forward(self, token_ids):
        vectors = self.weight[token_ids]
        if self.scale:
            vectors = vectors * math.sqrt(self.embedding_dimension)
        return vectors


## Sinusoidal positional encoding

Generates sine values for even dimensions and cosine values for odd dimensions, then adds them to token embeddings.

In [10]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embedding_dimension, max_length):
        super().__init__()

        if embedding_dimension <= 0:
            raise ValueError("embedding_dimension must be positive.")
        if max_length <= 0:
            raise ValueError("max_length must be positive.")

        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        even_dimensions = torch.arange(
            0,
            embedding_dimension,
            2,
            dtype=torch.float32,
        )
        angle_rates = torch.exp(
            -math.log(10000.0) * even_dimensions / embedding_dimension
        )
        angles = positions * angle_rates.unsqueeze(0)

        encoding = torch.zeros(
            max_length,
            embedding_dimension,
            dtype=torch.float32,
        )
        encoding[:, 0::2] = torch.sin(angles)

        cosine_width = encoding[:, 1::2].shape[1]
        encoding[:, 1::2] = torch.cos(angles[:, :cosine_width])

        # A buffer moves with the module but is not a trainable parameter.
        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
            persistent=True,
        )

    def forward(self, token_embeddings):
        sequence_length = token_embeddings.shape[1]
        if sequence_length > self.encoding.shape[1]:
            raise ValueError("Input is longer than configured max_length.")

        positional_values = self.encoding[:, :sequence_length]
        positional_values = positional_values.to(
            device=token_embeddings.device,
            dtype=token_embeddings.dtype,
        )
        return token_embeddings + positional_values


## Inspect sine/cosine values

Displays a small slice to show that each position receives a distinct deterministic pattern.

In [11]:
positional_encoder = SinusoidalPositionalEncoding(
    embedding_dimension=config.embedding_dimension,
    max_length=config.max_sequence_length,
)

print(
    positional_encoder.encoding[0, :5, :8]
)
print(
    "Positional table shape:",
    tuple(positional_encoder.encoding.shape),
)


tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.7617,  0.6479,  0.6816,  0.7318,  0.6047,  0.7965],
        [ 0.9093, -0.4161,  0.9870, -0.1604,  0.9975,  0.0709,  0.9632,  0.2687],
        [ 0.1411, -0.9900,  0.5173, -0.8558,  0.7783, -0.6279,  0.9296, -0.3685],
        [-0.7568, -0.6536, -0.3167, -0.9485,  0.1415, -0.9899,  0.5176, -0.8556]])
Positional table shape: (1, 32, 128)


## Position-encoded export pipeline

Combines token and position vectors in batches, zeros padded positions, and stores the resulting tensors on CPU.

In [12]:
class PositionalEmbeddingExporter:
    def __init__(
        self,
        source_embedding,
        target_embedding,
        positional_encoder,
        device,
        save_dtype=torch.float16,
    ):
        self.source_embedding = source_embedding.to(device).eval()
        self.target_embedding = target_embedding.to(device).eval()
        self.positional_encoder = positional_encoder.to(device).eval()
        self.device = device
        self.save_dtype = save_dtype

    @torch.no_grad()
    def encode(self, dataloader):
        output = {
            "source_ids": [],
            "target_ids": [],
            "source_padding_mask": [],
            "target_padding_mask": [],
            "source_position_encoded": [],
            "target_position_encoded": [],
        }

        for batch in dataloader:
            source_ids = batch["source_ids"].to(self.device)
            target_ids = batch["target_ids"].to(self.device)
            source_mask = batch["source_padding_mask"].to(self.device)
            target_mask = batch["target_padding_mask"].to(self.device)

            source_vectors = self.positional_encoder(
                self.source_embedding(source_ids)
            )
            target_vectors = self.positional_encoder(
                self.target_embedding(target_ids)
            )

            # Positional values would otherwise make PAD locations nonzero.
            source_vectors = source_vectors.masked_fill(
                source_mask.unsqueeze(-1), 0.0
            )
            target_vectors = target_vectors.masked_fill(
                target_mask.unsqueeze(-1), 0.0
            )

            output["source_ids"].append(source_ids.cpu())
            output["target_ids"].append(target_ids.cpu())
            output["source_padding_mask"].append(source_mask.cpu())
            output["target_padding_mask"].append(target_mask.cpu())
            output["source_position_encoded"].append(
                source_vectors.to(self.save_dtype).cpu()
            )
            output["target_position_encoded"].append(
                target_vectors.to(self.save_dtype).cpu()
            )

        return {
            name: torch.cat(tensors, dim=0)
            for name, tensors in output.items()
        }


## Initialize embedding and exporter objects

Creates the English and Bangla lookup layers from Drive matrices and connects them to one positional encoder.

In [13]:
source_embedding = PretrainedTokenEmbedding(
    embedding_store.english_matrix,
    trainable=False,
    scale=config.scale_token_embeddings,
)
target_embedding = PretrainedTokenEmbedding(
    embedding_store.bangla_matrix,
    trainable=False,
    scale=config.scale_token_embeddings,
)

save_dtype = torch.float16 if config.save_as_float16 else torch.float32

exporter = PositionalEmbeddingExporter(
    source_embedding=source_embedding,
    target_embedding=target_embedding,
    positional_encoder=positional_encoder,
    device=device,
    save_dtype=save_dtype,
)


## Encode and save all splits

Processes train, validation, and test data and saves fixed-length IDs, masks, and positional embeddings to Drive.

In [14]:
saved_paths = {}

for split_name, records in tokenized_records.items():
    dataset = FixedLengthTokenDataset(
        records,
        max_length=config.max_sequence_length,
        padding_id=config.padding_id,
    )
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=False,
    )

    encoded_split = exporter.encode(dataloader)
    encoded_split["metadata"] = {
        "split": split_name,
        "embedding_dimension": config.embedding_dimension,
        "max_sequence_length": config.max_sequence_length,
        "padding_id": config.padding_id,
        "saved_dtype": str(save_dtype),
        "token_embeddings_scaled": config.scale_token_embeddings,
    }

    save_path = output_dir / f"{split_name}_position_encoded.pt"
    torch.save(encoded_split, save_path)
    saved_paths[split_name] = save_path

    print(split_name, tuple(encoded_split["source_position_encoded"].shape))
    print("Saved:", save_path)

metadata_path = output_dir / "positional_encoding_metadata.json"
metadata_path.write_text(
    json.dumps(
        {
            "config": asdict(config),
            "saved_files": {
                split_name: str(path)
                for split_name, path in saved_paths.items()
            },
            "formula": {
                "even_dimensions": "sin(position / 10000^(dimension/d_model))",
                "odd_dimensions": "cos(position / 10000^((dimension-1)/d_model))",
            },
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("Metadata saved:", metadata_path)


train (1800, 32, 128)
Saved: /content/drive/MyDrive/Transformer/position_encoded_embeddings/128d/train_position_encoded.pt
validation (100, 32, 128)
Saved: /content/drive/MyDrive/Transformer/position_encoded_embeddings/128d/validation_position_encoded.pt
test (100, 32, 128)
Saved: /content/drive/MyDrive/Transformer/position_encoded_embeddings/128d/test_position_encoded.pt
Metadata saved: /content/drive/MyDrive/Transformer/position_encoded_embeddings/128d/positional_encoding_metadata.json


## Reload and validate a saved split

Reloads the training tensors from Drive and checks shapes, dimensions, and zeroed padding positions.

In [15]:
def load_saved_position_encoded(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


loaded_train = load_saved_position_encoded(saved_paths["train"])

source_vectors = loaded_train["source_position_encoded"]
source_mask = loaded_train["source_padding_mask"]

assert source_vectors.ndim == 3
assert source_vectors.shape[1] == config.max_sequence_length
assert source_vectors.shape[2] == config.embedding_dimension
assert torch.count_nonzero(source_vectors[source_mask]) == 0

print("Loaded source shape:", tuple(source_vectors.shape))
print(
    "Loaded target shape:",
    tuple(loaded_train["target_position_encoded"].shape),
)
print("Saved positional embeddings validated successfully.")


Loaded source shape: (1800, 32, 128)
Loaded target shape: (1800, 32, 128)
Saved positional embeddings validated successfully.


## Compare the same token at two positions

Demonstrates that identical token vectors become different after adding distinct position vectors.

In [16]:
sample_token_id = loaded_train["source_ids"][0, 1]
token_vector = source_embedding(
    sample_token_id.view(1, 1).to(device)
).cpu()[0, 0]

position_zero = positional_encoder.encoding[0, 0].cpu()
position_one = positional_encoder.encoding[0, 1].cpu()

encoded_at_zero = token_vector + position_zero
encoded_at_one = token_vector + position_one

print("Token ID:", int(sample_token_id))
print(
    "Vectors equal before changing position:",
    torch.allclose(token_vector, token_vector),
)
print(
    "Position-encoded vectors equal:",
    torch.allclose(encoded_at_zero, encoded_at_one),
)


Token ID: 43
Vectors equal before changing position: True
Position-encoded vectors equal: False
